# Milestone 3 — Analisis Temporal dan Advanced EDA Indonesia News

**Nama:** Naufal Zainul Zaki  
**NIM:** 202310370311217  
**GitHub:** Falz4i

Mengikuti [template terbaru dosen](https://github.com/UMM-GURU/tugas1-abd-2026/blob/main/notebooks/03_eda_and_insights.ipynb), diperiksa 26 Oktober 2026. Gunakan hasil cleaning revisi 2026-10-26.

**Batas interpretasi:** unit analisis adalah baris artikel setelah deduplikasi sumber + judul + hash isi, bukan jumlah peristiwa atau pembaca. Revisi isi dan duplikat semantik masih mungkin ada. Arsip tiap media tidak selalu lengkap; 2014 dan 2020 parsial. Timestamp dibaca sebagai epoch UTC, belum divalidasi terhadap waktu lokal redaksi. Korelasi bersifat deskriptif dan tidak membuktikan sebab akibat.


In [1]:
from pathlib import Path
from IPython.display import display, Markdown
import duckdb
import json
import plotly.graph_objects as go
import polars as pl
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "plotly_mimetype"
pio.templates.default = "plotly_white"
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "indonesia_news_clean.parquet"
FIGURE_DIR = PROJECT_ROOT / "output" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.exists():
    raise FileNotFoundError("Jalankan 02_data_cleaning.ipynb terlebih dahulu.")
clean_scan = pl.scan_parquet(DATA_PATH)
print('Schema hasil cleaning:', clean_scan.collect_schema())
if 'content_sha256' not in clean_scan.collect_schema():
    raise ValueError('Jalankan notebook cleaning revisi 26 Oktober 2026 terlebih dahulu.')
metadata = json.loads((DATA_PATH.parent / 'cleaning_metadata.json').read_text(encoding='utf-8'))
assert metadata['pipeline_version'] == '2026-10-26'
con = duckdb.connect()
con.execute("SET memory_limit='2GB'")
con.execute('SET threads=2')
con.from_parquet(str(DATA_PATH)).create_view("news", replace=True)
COLORS = {"CNN Indonesia": "#d62728", "Tempo": "#1f77b4"}
print("Sumber:", DATA_PATH)
print("DuckDB:", duckdb.__version__)

Schema hasil cleaning: Schema({'source': String, 'source_file': String, 'source_row': UInt32, 'published_at': Datetime(time_unit='us', time_zone=None), 'title': String, 'tags': String, 'content': String, 'title_length': UInt32, 'content_length': UInt32, 'tags_missing': Boolean, 'content_sha256': String, 'content_length_outlier': Boolean, 'iqr_lower': Float64, 'iqr_upper': Float64})
Sumber: /home/jovyan/work/data/processed/indonesia_news_clean.parquet
DuckDB: 1.5.5


## 1. Pemeriksaan data dan pertanyaan analisis

Bagaimana jumlah artikel berubah tiap bulan? Bagaimana komposisi sumber, tag, panjang teks, dan pola hari publikasi? DuckDB melakukan agregasi langsung dari Parquet agar teks seluruh dataset tidak perlu dimuat ke memori.

In [2]:
overview = con.sql("""
SELECT COUNT(*) AS article_count, MIN(published_at) AS first_date,
       MAX(published_at) AS last_date, COUNT(DISTINCT source) AS source_count
FROM news
""").pl()
if overview.item(0, "article_count") == 0:
    raise ValueError("Dataset bersih kosong.")
display(overview)
quality = con.sql("""
SELECT COUNT(*) FILTER (WHERE title IS NULL OR TRIM(title) = '') AS invalid_title,
       COUNT(*) FILTER (WHERE content IS NULL OR content_length < 100) AS invalid_content,
       COUNT(*) FILTER (WHERE published_at IS NULL) AS invalid_date
FROM news
""").pl()
display(quality)
assert all(value == 0 for value in quality.row(0)), "Kualitas data wajib belum terpenuhi."

def show_and_save(fig, filename):
    fig.update_layout(margin=dict(t=80, l=60, r=30, b=70))
    fig.write_html(FIGURE_DIR / filename, include_plotlyjs=True, full_html=True)
    fig.show()

total_articles = overview.item(0, "article_count")
assert total_articles == metadata["rows_after"]

article_count,first_date,last_date,source_count
i64,datetime[μs],datetime[μs],i64
848028,2014-08-04 08:35:00,2020-05-31 16:52:00,2


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

invalid_title,invalid_content,invalid_date
i64,i64,i64
0,0,0


## 2. Jumlah artikel per bulan

Grafik 1 memungkinkan zoom dan hover untuk melihat perubahan jumlah artikel tiap media. Bulan tanpa baris diisi nol untuk membedakannya dari titik yang hilang; nol tetap dapat berarti arsip tidak lengkap.

In [3]:
monthly = con.sql("""
WITH bounds AS (
    SELECT MIN(date_trunc('month', published_at)) AS lo,
           MAX(date_trunc('month', published_at)) AS hi FROM news
), months AS (
    SELECT unnest(generate_series(lo, hi, INTERVAL '1 month')) AS month FROM bounds
), sources AS (SELECT DISTINCT source FROM news),
counts AS (
    SELECT date_trunc('month', published_at) AS month, source, COUNT(*) AS article_count
    FROM news GROUP BY 1, 2
)
SELECT m.month, s.source, COALESCE(c.article_count, 0) AS article_count
FROM months m CROSS JOIN sources s
LEFT JOIN counts c ON m.month = c.month AND s.source = c.source
ORDER BY m.month, s.source
""").pl()
fig = px.line(monthly.to_pandas(), x="month", y="article_count", color="source",
              color_discrete_map=COLORS, title="1. Jumlah artikel per bulan",
              labels={"month": "Bulan (UTC)", "article_count": "Jumlah artikel", "source": "Media"})
fig.update_layout(hovermode="x unified")
fig.update_xaxes(rangeslider_visible=True)
show_and_save(fig, "01_monthly_articles.html")

## 3. Jumlah artikel per tahun

Grafik 2 membandingkan arsip tahunan per media. Tahun 2014 dan 2020 diberi keterangan karena periode pengamatan parsial; jangan menafsirkan penurunan 2020 sebagai penurunan satu tahun penuh.

In [4]:
annual = con.sql("""
SELECT year(published_at) AS year, source, COUNT(*) AS article_count
FROM news GROUP BY 1, 2 ORDER BY 1, 2
""").pl().with_columns(
    pl.when(pl.col("year").is_in([2014, 2020]))
    .then(pl.col("year").cast(pl.String) + pl.lit(" (parsial)"))
    .otherwise(pl.col("year").cast(pl.String)).alias("year_label")
)
fig = px.bar(annual.to_pandas(), x="year_label", y="article_count", color="source",
             barmode="group", color_discrete_map=COLORS,
             title="2. Jumlah artikel per tahun — 2014 dan 2020 parsial",
             labels={"year_label": "Tahun", "article_count": "Jumlah artikel", "source": "Media"})
show_and_save(fig, "02_annual_articles.html")

## 4. Komposisi sumber berita

Grafik 3 menunjukkan proporsi baris pada dataset bersih. Proporsi ini menggambarkan arsip dataset dan bukan pangsa pasar media.

In [5]:
sources = con.sql("""
SELECT source, COUNT(*) AS article_count FROM news GROUP BY source
ORDER BY article_count DESC
""").pl()
fig = px.pie(sources.to_pandas(), names="source", values="article_count", hole=0.45,
             color="source", color_discrete_map=COLORS,
             title="3. Komposisi sumber pada dataset bersih")
fig.update_traces(textinfo="percent+label")
show_and_save(fig, "03_source_share.html")

## 5. Hari dan jam publikasi

Grafik 4 menampilkan jumlah artikel per hari dalam minggu dan jam UTC. Heatmap mengukur volume arsip, bukan produktivitas yang dinormalisasi per hari. Pola jam perlu diperiksa ulang setelah zona waktu sumber dikonfirmasi.

In [6]:
weekday_names = ["Senin", "Selasa", "Rabu", "Kamis", "Jumat", "Sabtu", "Minggu"]
hourly = con.sql("""
SELECT isodow(published_at) AS weekday, hour(published_at) AS hour,
       COUNT(*) AS article_count
FROM news GROUP BY 1, 2 ORDER BY 1, 2
""").pl()
matrix = [[0] * 24 for _ in range(7)]
for row in hourly.iter_rows(named=True):
    matrix[row["weekday"] - 1][row["hour"]] = row["article_count"]
fig = px.imshow(matrix, x=list(range(24)), y=weekday_names, aspect="auto",
                color_continuous_scale="Blues", title="4. Volume artikel menurut hari dan jam UTC",
                labels={"x": "Jam UTC", "y": "Hari", "color": "Jumlah artikel"})
fig.update_traces(hovertemplate="%{y}, jam %{x} UTC<br>Artikel: %{z:,}<extra></extra>")
show_and_save(fig, "04_weekday_hour.html")

## 6. Distribusi dan pencilan panjang isi

Grafik 5 adalah box plot dengan kuartil yang dihitung dari seluruh data. Whisker adalah nilai aktual terjauh dalam rentang 1,5 IQR. Titik pencilan tidak digambar satu per satu; jumlah flag cleaning dan panjang maksimum ditampilkan pada tabel. Flag cleaning menggunakan baseline sebelum deduplikasi, sehingga dapat berbeda dengan IQR hasil akhir. Teks panjang tetap dipertahankan.

In [7]:
length_box = con.sql("""
SELECT source, quantile_cont(content_length, 0.25) AS q1,
       median(content_length) AS median, quantile_cont(content_length, 0.75) AS q3,
       min(content_length) AS min_length, max(content_length) AS max_length
FROM news GROUP BY source ORDER BY source
""").pl()
fig = go.Figure()
for row in length_box.iter_rows(named=True):
    iqr = row['q3'] - row['q1']
    lower = row['q1'] - 1.5 * iqr
    upper = row['q3'] + 1.5 * iqr
    whiskers = con.execute(
        'SELECT min(content_length), max(content_length) FROM news WHERE source=? AND content_length BETWEEN ? AND ?',
        [row['source'], lower, upper]
    ).fetchone()
    fig.add_trace(go.Box(name=row['source'], x=[row['source']], q1=[row['q1']], median=[row['median']], q3=[row['q3']],
                         lowerfence=[whiskers[0]], upperfence=[whiskers[1]],
                         marker_color=COLORS[row['source']]))
fig.update_layout(title='5. Distribusi panjang isi menurut media', yaxis_title='Panjang isi (karakter)')
show_and_save(fig, '05_content_length.html')
display(length_box)
long_count = con.sql('SELECT count(*) FROM news WHERE content_length >= 5000').fetchone()[0]
display(con.sql('SELECT source, sum(CAST(content_length_outlier AS INTEGER)) AS flagged_outliers FROM news GROUP BY source').pl())


source,q1,median,q3,min_length,max_length
str,f64,f64,f64,u32,u32
"""CNN Indonesia""",1677.0,2044.0,2582.0,100,30074
"""Tempo""",1427.0,1774.0,2231.0,100,52721


source,flagged_outliers
str,"decimal[38,0]"
"""CNN Indonesia""",17401
"""Tempo""",23433


## 7. Tag yang paling sering muncul

Grafik 6 memisahkan tag dengan koma, merapikan spasi, dan menyamakan huruf kecil. Tag sama yang berulang pada satu artikel hanya dihitung sekali. Artikel tanpa tag dikecualikan dari grafik ini. Tag bukan klasifikasi topik yang seragam antar-media; satu artikel dapat memiliki lebih dari satu tag.

In [8]:
top_tags = con.sql("""
WITH tag_counts AS (
    SELECT unnest(list_distinct(list_transform(
        string_split(tags, ','), x -> lower(trim(x))
    ))) AS tag
    FROM news WHERE tags <> 'Tidak diketahui'
)
SELECT tag, COUNT(*) AS article_count FROM tag_counts
WHERE tag IS NOT NULL AND tag <> ''
GROUP BY tag ORDER BY article_count DESC, tag LIMIT 20
""").pl()
fig = px.bar(top_tags.to_pandas(), x="article_count", y="tag", orientation="h",
             title="6. Dua puluh tag paling sering muncul",
             labels={"article_count": "Artikel yang memuat tag", "tag": "Tag"})
fig.update_yaxes(categoryorder="total ascending")
show_and_save(fig, "06_top_tags.html")

## 8. Perbandingan panjang artikel antar-media

Grafik 7 membandingkan kuartil panjang isi. Kuartil diperoleh dari seluruh baris dengan SQL, tanpa mengirim seluruh teks ke Plotly.

In [9]:
length_stats = con.sql("""
SELECT source, quantile_cont(content_length, 0.25) AS q25,
       median(content_length) AS median_length,
       quantile_cont(content_length, 0.75) AS q75,
       MAX(content_length) AS max_length
FROM news GROUP BY source ORDER BY source
""").pl()
quartiles = length_stats.unpivot(
    on=["q25", "median_length", "q75"], index="source",
    variable_name="statistic", value_name="characters"
)
fig = px.bar(quartiles.to_pandas(), x="source", y="characters", color="statistic",
             barmode="group", title="7. Kuartil panjang isi menurut media",
             labels={"source": "Media", "characters": "Panjang isi (karakter)", "statistic": "Statistik"},
             category_orders={"statistic": ["q25", "median_length", "q75"]})
show_and_save(fig, "07_length_quartiles.html")
display(length_stats)

source,q25,median_length,q75,max_length
str,f64,f64,f64,u32
"""CNN Indonesia""",1677.0,2044.0,2582.0,30074
"""Tempo""",1427.0,1774.0,2231.0,52721


## 9. Temporal windowing — moving average dan perubahan bulanan

Agregasi `date_trunc` sebelumnya dilengkapi SQL window `OVER`. Moving average memakai bulan berjalan dan dua bulan sebelumnya **per media**, setelah kalender bulanan dilengkapi. MA baru ditampilkan jika ketiga bulan berada dalam rentang observasi media. Perubahan bulanan menggunakan `LAG`; pembagi nol menghasilkan null. Bulan parsial diberi flag agar tidak menjadi dasar rekomendasi pertumbuhan.

In [10]:
con.register('monthly_counts', monthly)
windowed = con.sql("""
WITH coverage AS (
 SELECT source, date_trunc('month', min(published_at)) AS first_month,
        date_trunc('month', max(published_at)) AS last_month FROM news GROUP BY source
), series AS (
 SELECT m.*, (m.month > c.first_month AND m.month < c.last_month) AS complete_period
 FROM monthly_counts m JOIN coverage c USING(source)
 WHERE m.month BETWEEN c.first_month AND c.last_month
), windows AS (
 SELECT *, AVG(article_count) OVER w AS ma3_value,
        COUNT(*) OVER w AS window_size,
        MIN(CAST(complete_period AS INTEGER)) OVER w AS complete_window,
        LAG(article_count) OVER (PARTITION BY source ORDER BY month) AS previous_count,
        LAG(complete_period) OVER (PARTITION BY source ORDER BY month) AS previous_complete
 FROM series WINDOW w AS (PARTITION BY source ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
)
SELECT *, CASE WHEN window_size=3 AND complete_window=1 THEN ma3_value END AS ma3,
 CASE WHEN complete_period AND previous_complete THEN
    100.0*(article_count - previous_count)/NULLIF(previous_count, 0) END AS month_change_pct
FROM windows ORDER BY source, month
""").pl()
fig = go.Figure()
for source in COLORS:
    frame = windowed.filter(pl.col('source') == source)
    fig.add_trace(go.Scatter(x=frame['month'].to_list(), y=frame['article_count'].to_list(),
                            name=f'{source} — bulanan', line=dict(color=COLORS[source], width=1)))
    fig.add_trace(go.Scatter(x=frame['month'].to_list(), y=frame['ma3'].to_list(),
                            name=f'{source} — MA3', line=dict(color=COLORS[source], width=3, dash='dash')))
fig.update_layout(title='8. Tren bulanan dan rata-rata bergerak tiga bulan',
                  xaxis_title='Bulan (UTC)', yaxis_title='Jumlah artikel', hovermode='x unified')
fig.update_xaxes(rangeslider_visible=True)
show_and_save(fig, '08_temporal_window.html')
display(windowed.select('source', 'month', 'article_count', 'ma3', 'month_change_pct').tail(12))


source,month,article_count,ma3,month_change_pct
str,datetime[μs],i64,f64,f64
"""Tempo""",2019-06-01 00:00:00,7472,8098.666667,-14.10507
"""Tempo""",2019-07-01 00:00:00,9509,8560.0,27.261777
"""Tempo""",2019-08-01 00:00:00,9753,8911.333333,2.56599
"""Tempo""",2019-09-01 00:00:00,10047,9769.666667,3.014457
"""Tempo""",2019-10-01 00:00:00,10651,10150.333333,6.011745
…,…,…,…,…
"""Tempo""",2020-01-01 00:00:00,10582,10006.666667,10.436235
"""Tempo""",2020-02-01 00:00:00,10121,10095.0,-4.356454
"""Tempo""",2020-03-01 00:00:00,10748,10483.666667,6.19504


## 10. Korelasi numerik dan analisis multidimensi

Korelasi Pearson antara panjang judul dan isi dihitung pada **seluruh baris**, secara keseluruhan dan per media. Scatter hanya menampilkan sampel deterministik 2.000 baris per media agar ringan; sampel seimbang itu tidak mewakili proporsi media sebenarnya. Warna membedakan media, facet membedakan tahun. Hubungan lemah tetap merupakan hasil dan tidak dipaksakan menjadi signifikan. Sumbu log membantu melihat sebaran, sedangkan nilai korelasi memakai panjang asli.

Ini memenuhi alternatif analisis multidimensi (panjang judul, isi, sumber, tahun). Dataset tidak memiliki koordinat terverifikasi sehingga peta tidak dibuat.

In [11]:
correlation = con.sql("""
SELECT 'Semua media' AS source, count(*) AS n,
       corr(title_length, content_length) AS pearson_r FROM news
UNION ALL
SELECT source, count(*), corr(title_length, content_length) FROM news GROUP BY source
""").pl()
display(correlation)
sample = con.sql("""
WITH ranked AS (
 SELECT source, year(published_at)::VARCHAR AS year, title_length, content_length,
        row_number() OVER (PARTITION BY source ORDER BY content_sha256, title, published_at, source_file, source_row) AS rank
 FROM news
)
SELECT source, year, title_length, content_length FROM ranked WHERE rank<=2000
ORDER BY source, rank
""").pl()
fig = px.scatter(sample.to_pandas(), x='title_length', y='content_length', color='source',
                 facet_col='year', facet_col_wrap=4, opacity=0.4, log_y=True,
                 color_discrete_map=COLORS, category_orders={'year': sorted(sample['year'].unique().to_list())},
                 title='9. Hubungan panjang judul dan isi menurut media dan tahun (sampel)',
                 labels={'title_length':'Panjang judul (karakter)', 'content_length':'Panjang isi (karakter)', 'source':'Media', 'year':'Tahun'})
fig.update_layout(height=650)
show_and_save(fig, '09_multidimensional_correlation.html')


source,n,pearson_r
str,i64,f64
"""Semua media""",848028,-0.05854
"""CNN Indonesia""",332444,0.030484
"""Tempo""",515584,-0.059956


## 11. Ringkasan angka hasil analisis

Semua angka dihitung ulang dari hasil cleaning revisi. Temuan interpretatif mengikuti ringkasan ini.

In [12]:
peak_month = monthly.group_by("month").agg(pl.col("article_count").sum()).sort(
    ["article_count", "month"], descending=[True, False]
).row(0, named=True)
leader = sources.row(0, named=True)
full_years = annual.filter(pl.col("year").is_between(2015, 2019)).group_by("year").agg(
    pl.col("article_count").sum()
).sort(["article_count", "year"], descending=[True, False])
peak_year = full_years.row(0, named=True)
weekday_totals = hourly.group_by("weekday").agg(pl.col("article_count").sum()).sort(
    ["article_count", "weekday"], descending=[True, False]
)
peak_day = weekday_totals.row(0, named=True)
text_leader = length_stats.sort("median_length", descending=True).row(0, named=True)

insights = [
    f"Dataset bersih mencakup **{total_articles:,} artikel** dari "
    f"**{overview.item(0, 'first_date'):%d-%m-%Y}** sampai **{overview.item(0, 'last_date'):%d-%m-%Y}**.",
    f"**{leader['source']}** menyumbang arsip terbanyak, yaitu **{leader['article_count']:,} artikel "
    f"({leader['article_count'] / total_articles * 100:.2f}%)**. Ini proporsi dataset, bukan pangsa pasar.",
    f"Bulan dengan jumlah artikel tertinggi adalah **{peak_month['month']:%Y-%m}**, "
    f"dengan **{peak_month['article_count']:,} artikel** dari kedua media.",
    f"Di antara tahun penuh 2015–2019, **{peak_year['year']}** memiliki arsip terbanyak: "
    f"**{peak_year['article_count']:,} artikel**.",
    f"Akumulasi publikasi tertinggi jatuh pada **{weekday_names[peak_day['weekday'] - 1]}**, "
    f"sebanyak **{peak_day['article_count']:,} artikel** berdasarkan timestamp UTC.",
    f"**{text_leader['source']}** memiliki median isi lebih panjang: "
    f"**{text_leader['median_length']:,.0f} karakter**.",
    f"Sebanyak **{long_count:,} artikel ({long_count / total_articles * 100:.2f}%)** "
    f"memiliki isi minimal 5.000 karakter. Teks panjang dipertahankan untuk analisis.",
]
if top_tags.height:
    first_tag = top_tags.row(0, named=True)
    insights.append(f"Tag paling sering muncul adalah **{first_tag['tag']}**, "
                    f"tercatat pada **{first_tag['article_count']:,} artikel**. "
                    "Tag dihitung setelah pemisahan koma dan normalisasi huruf.")
display(Markdown("\n\n".join(f"{i}. {text}" for i, text in enumerate(insights, 1))))



1. Dataset bersih mencakup **848,028 artikel** dari **04-08-2014** sampai **31-05-2020**.

2. **Tempo** menyumbang arsip terbanyak, yaitu **515,584 artikel (60.80%)**. Ini proporsi dataset, bukan pangsa pasar.

3. Bulan dengan jumlah artikel tertinggi adalah **2020-03**, dengan **17,043 artikel** dari kedua media.

4. Di antara tahun penuh 2015–2019, **2019** memiliki arsip terbanyak: **176,524 artikel**.

5. Akumulasi publikasi tertinggi jatuh pada **Rabu**, sebanyak **145,301 artikel** berdasarkan timestamp UTC.

6. **CNN Indonesia** memiliki median isi lebih panjang: **2,044 karakter**.

7. Sebanyak **11,256 artikel (1.33%)** memiliki isi minimal 5.000 karakter. Teks panjang dipertahankan untuk analisis.

8. Tag paling sering muncul adalah **jokowi**, tercatat pada **29,323 artikel**. Tag dihitung setelah pemisahan koma dan normalisasi huruf.

## 12. Lima insight analitik utama dan rekomendasi

Setiap insight menghubungkan hasil numerik, grafik pendukung, implikasi, dan batas interpretasi. Tidak ada klaim kausal atau uji signifikansi statistik.

In [13]:
r = correlation.filter(pl.col('source') == 'Semua media').item(0, 'pearson_r')
relation = 'tidak dapat dihitung' if r is None else ('lemah' if abs(r)<0.3 else 'sedang' if abs(r)<0.7 else 'kuat')
r_text = 'tidak tersedia' if r is None else f'{r:.3f}'
biggest_change = windowed.filter(pl.col('month_change_pct').is_not_null()).sort(
    ['month_change_pct', 'source', 'month'], descending=[True, False, False]
).row(0, named=True)
by_source = {row['source']: row for row in length_stats.iter_rows(named=True)}
median_gap = by_source['CNN Indonesia']['median_length'] - by_source['Tempo']['median_length']
findings = [
    f"**Tren waktu (grafik 1 dan 8).** Puncak bulanan gabungan terjadi pada {peak_month['month']:%Y-%m} "
    f"({peak_month['article_count']:,} artikel). Kenaikan bulanan terbesar pada periode interior observasi adalah "
    f"{biggest_change['source']} pada {biggest_change['month']:%Y-%m}: {biggest_change['month_change_pct']:.2f}%, "
    f"dari {biggest_change['previous_count']:,} menjadi {biggest_change['article_count']:,} artikel. "
    "Persentase ekstrem ini harus dibaca bersama penyebutnya; basis bulan sebelumnya yang kecil dapat "
    "menghasilkan lonjakan persentase sangat besar dan mengindikasikan arsip yang tidak merata. "
    "MA3 membantu membedakan perubahan satu bulan dari kecenderungan beberapa bulan. Perubahan dapat disebabkan cakupan arsip, bukan hanya aktivitas redaksi; tidak diatribusikan ke peristiwa tertentu tanpa verifikasi.",
    f"**Distribusi dan anomali (grafik 5).** {long_count:,} artikel ({100*long_count/total_articles:.2f}%) "
    "memiliki isi sedikitnya 5.000 karakter. Ekornya perlu diperhatikan saat merangkum panjang isi; median dan IQR "
    "lebih informatif daripada rata-rata saja. Panjang tinggi merupakan kandidat pemeriksaan, bukan otomatis kesalahan.",
    f"**Korelasi (grafik 9).** Pearson seluruh data = {r_text}, dengan kekuatan deskriptif {relation}. "
    "Bandingkan tabel per media untuk melihat kemungkinan pengaruh komposisi. Panjang judul saja tidak cukup untuk "
    "menyimpulkan kualitas, popularitas, atau sebab perubahan panjang isi; scatter sampel bukan hasil uji signifikansi.",
    f"**Segmen media (grafik 3 dan 7).** {leader['source']} menyumbang {100*leader['article_count']/total_articles:.2f}% arsip. "
    f"Selisih median panjang isi CNN Indonesia dikurangi Tempo adalah {median_gap:,.0f} karakter. "
    "Ketimpangan jumlah baris dan distribusi panjang menunjukkan perlunya melaporkan hasil per media; agregat gabungan lebih dipengaruhi sumber dengan arsip lebih banyak.",
    "**Rekomendasi praktis.** Untuk analisis lanjutan, bandingkan media pada bulan observasi yang sama, gunakan MA3 "
    "untuk memantau tren, serta tampilkan median/IQR panjang isi. Audit sampel teks pendek dan pencilan sebelum "
    "mengubah threshold cleaning. Gunakan sampel seimbang untuk perbandingan eksploratif, tetapi hitung proporsi dari "
    "seluruh data. Konfirmasi timestamp dan cakupan pengambilan data sebelum membuat rekomendasi jadwal publikasi."
]
display(Markdown('\n\n'.join(f'{i}. {s}' for i,s in enumerate(findings, 1))))
for name, frame in {'monthly': monthly, 'windowed': windowed, 'correlation': correlation,
                    'length_stats': length_stats, 'sources': sources}.items():
    frame.write_csv(PROJECT_ROOT / 'output' / f'{name}.csv')
con.close()
print('Selesai. Grafik HTML:', FIGURE_DIR)


1. **Tren waktu (grafik 1 dan 8).** Puncak bulanan gabungan terjadi pada 2020-03 (17,043 artikel). Kenaikan bulanan terbesar pada periode interior observasi adalah CNN Indonesia pada 2015-01: 46416.67%, dari 12 menjadi 5,582 artikel. Persentase ekstrem ini harus dibaca bersama penyebutnya; basis bulan sebelumnya yang kecil dapat menghasilkan lonjakan persentase sangat besar dan mengindikasikan arsip yang tidak merata. MA3 membantu membedakan perubahan satu bulan dari kecenderungan beberapa bulan. Perubahan dapat disebabkan cakupan arsip, bukan hanya aktivitas redaksi; tidak diatribusikan ke peristiwa tertentu tanpa verifikasi.

2. **Distribusi dan anomali (grafik 5).** 11,256 artikel (1.33%) memiliki isi sedikitnya 5.000 karakter. Ekornya perlu diperhatikan saat merangkum panjang isi; median dan IQR lebih informatif daripada rata-rata saja. Panjang tinggi merupakan kandidat pemeriksaan, bukan otomatis kesalahan.

3. **Korelasi (grafik 9).** Pearson seluruh data = -0.059, dengan kekuatan deskriptif lemah. Bandingkan tabel per media untuk melihat kemungkinan pengaruh komposisi. Panjang judul saja tidak cukup untuk menyimpulkan kualitas, popularitas, atau sebab perubahan panjang isi; scatter sampel bukan hasil uji signifikansi.

4. **Segmen media (grafik 3 dan 7).** Tempo menyumbang 60.80% arsip. Selisih median panjang isi CNN Indonesia dikurangi Tempo adalah 270 karakter. Ketimpangan jumlah baris dan distribusi panjang menunjukkan perlunya melaporkan hasil per media; agregat gabungan lebih dipengaruhi sumber dengan arsip lebih banyak.

5. **Rekomendasi praktis.** Untuk analisis lanjutan, bandingkan media pada bulan observasi yang sama, gunakan MA3 untuk memantau tren, serta tampilkan median/IQR panjang isi. Audit sampel teks pendek dan pencilan sebelum mengubah threshold cleaning. Gunakan sampel seimbang untuk perbandingan eksploratif, tetapi hitung proporsi dari seluruh data. Konfirmasi timestamp dan cakupan pengambilan data sebelum membuat rekomendasi jadwal publikasi.

Selesai. Grafik HTML: /home/jovyan/work/output/figures


## Pemetaan ke template dan reproduksibilitas

| Ketentuan | Bukti |
|---|---|
| Identitas | Header notebook dan `identitas.md` |
| Memuat data bersih dengan Polars | `scan_parquet` dan pemeriksaan schema, bagian awal |
| Temporal/windowing | MA3 dan LAG dengan SQL OVER, bagian 9 |
| Tren deret waktu dan range slider | Grafik 1 dan 8 |
| Distribusi/outlier | Box plot grafik 5 dan tabel flag IQR |
| Korelasi numerik | Pearson seluruh data dan scatter grafik 9 |
| Komposisi kategori | Grafik 3 serta bar chart tag grafik 6 |
| Siklus/perbandingan kelompok | Heatmap hari-jam grafik 4 dan kuartil grafik 7 |
| Spasial atau multidimensi | Scatter menurut media dan tahun, grafik 9 |
| Lima insight | Bagian 12 |

Jalankan notebook 02 dahulu, lalu Run All pada notebook 03 dan simpan outputnya. HTML grafik menyertakan Plotly agar dapat dibuka offline. Data besar dan HTML hasil regenerasi tidak di-commit; notebook memuat output grafik dan insight. Identitas: Naufal Zainul Zaki, 202310370311217, Falz4i.

**Bantuan AI:** Codex membantu penyesuaian kode, penjelasan dan verifikasi eksekusi. Pemahaman, pemeriksaan asumsi, serta interpretasi akhir tetap perlu dilakukan mahasiswa. Lihat AI Disclosure Statement di README.
